# 03 - ImageNet-100 Expert (ResNet-18)

Train a ResNet-18 classifier for ImageNet-100:
- Use pretrained ImageNet weights (transfer learning)
- Phase 1: Freeze backbone, train classifier only (10 epochs)
- Phase 2: Fine-tune entire network (40 epochs)
- Target: ~70-75% top-1 accuracy on clean images

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm

# Configuration
BASE_DIR = r"C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN"
os.chdir(BASE_DIR)

torch.set_num_threads(8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 100
NUM_WORKERS = 4

print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Data

In [ ]:
# Data paths
DATA_DIR = os.path.join(BASE_DIR, "imagenet100_data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

# ImageNet normalization (for pretrained ResNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transforms WITH normalization (for ResNet)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Datasets
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_transforms)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset):,} images ({len(train_loader)} batches)")
print(f"Val: {len(val_dataset):,} images ({len(val_loader)} batches)")
print(f"Classes: {len(train_dataset.classes)}")

## 2. Create ResNet-18 Model

In [ ]:
class ImageNet100Expert(nn.Module):
    """
    ResNet-18 modified for ImageNet-100 (100 classes).
    
    Uses pretrained ImageNet weights for transfer learning.
    Only the final fc layer is modified: 512 -> 100 classes.
    """
    
    def __init__(self, num_classes=100, pretrained=True, dropout=0.3):
        super(ImageNet100Expert, self).__init__()
        
        # Load pretrained ResNet-18
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.resnet = models.resnet18(weights=weights)
        
        # Replace final fc layer
        in_features = self.resnet.fc.in_features  # 512
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x):
        return self.resnet(x)
    
    def freeze_backbone(self):
        """Freeze all layers except the final fc."""
        for name, param in self.resnet.named_parameters():
            if 'fc' not in name:
                param.requires_grad = False
    
    def unfreeze_all(self):
        """Unfreeze all layers."""
        for param in self.resnet.parameters():
            param.requires_grad = True


# Create model
model = ImageNet100Expert(num_classes=NUM_CLASSES, pretrained=True, dropout=0.3).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"ImageNet100Expert (ResNet-18):")
print(f"  Total parameters: {total_params:,}")
print(f"  Output classes: {NUM_CLASSES}")

## 3. Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0.0
    correct = 0
    correct_top5 = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Top-5 accuracy
            _, top5_pred = outputs.topk(5, 1, True, True)
            correct_top5 += top5_pred.eq(labels.view(-1, 1).expand_as(top5_pred)).sum().item()
    
    return total_loss / len(loader), 100. * correct / total, 100. * correct_top5 / total

print("Training functions defined")

## 4. Phase 1: Train Classifier Only (Frozen Backbone)

In [ ]:
# Phase 1 configuration
PHASE1_EPOCHS = 10
PHASE1_LR = 1e-3

# Freeze backbone
model.freeze_backbone()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1: Classifier only")
print(f"  Trainable parameters: {trainable:,}")

# Loss with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer (only trainable params)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), 
                       lr=PHASE1_LR, weight_decay=1e-4)

print(f"  Epochs: {PHASE1_EPOCHS}")
print(f"  Learning rate: {PHASE1_LR}")

In [ ]:
print("\nPhase 1: Training classifier (backbone frozen)...")
print("="*60)

phase1_train_acc = []
phase1_val_acc = []

for epoch in range(PHASE1_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_top5 = evaluate(model, val_loader, criterion, device)
    
    phase1_train_acc.append(train_acc)
    phase1_val_acc.append(val_acc)
    
    print(f"Epoch {epoch+1}/{PHASE1_EPOCHS} | "
          f"Train: {train_acc:.1f}% | Val: {val_acc:.1f}% (Top-5: {val_top5:.1f}%)")

print(f"\nPhase 1 complete! Val accuracy: {val_acc:.1f}%")

## 5. Phase 2: Fine-tune Entire Network

In [ ]:
# Phase 2 configuration
PHASE2_EPOCHS = 40
PHASE2_LR = 1e-4  # Lower LR for fine-tuning

# Unfreeze all layers
model.unfreeze_all()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2: Full fine-tuning")
print(f"  Trainable parameters: {trainable:,}")

# New optimizer with lower LR
optimizer = optim.Adam(model.parameters(), lr=PHASE2_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS, eta_min=1e-6)

print(f"  Epochs: {PHASE2_EPOCHS}")
print(f"  Learning rate: {PHASE2_LR}")

In [ ]:
print("\nPhase 2: Fine-tuning entire network...")
print("="*60)

phase2_train_acc = []
phase2_val_acc = []
best_acc = 0.0

for epoch in range(PHASE2_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_top5 = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    
    phase2_train_acc.append(train_acc)
    phase2_val_acc.append(val_acc)
    
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{PHASE2_EPOCHS} | "
          f"Train: {train_acc:.1f}% | Val: {val_acc:.1f}% (Top-5: {val_top5:.1f}%) | LR: {lr:.1e}")
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(BASE_DIR, "models", "imagenet100_expert.pth"))
        print(f"  Saved best model (acc: {best_acc:.1f}%)")

print(f"\nPhase 2 complete!")
print(f"Best validation accuracy: {best_acc:.1f}%")

## 6. Training Curves

In [ ]:
# Combine phase 1 and phase 2 metrics
all_train_acc = phase1_train_acc + phase2_train_acc
all_val_acc = phase1_val_acc + phase2_val_acc

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(all_train_acc, 'b-', label='Train', linewidth=2)
plt.plot(all_val_acc, 'r-', label='Val', linewidth=2)
plt.axvline(x=PHASE1_EPOCHS-1, color='g', linestyle='--', label='Phase 2 start')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(phase2_val_acc, 'r-', linewidth=2)
plt.xlabel('Epoch (Phase 2)')
plt.ylabel('Validation Accuracy (%)')
plt.title('Phase 2 Validation Accuracy')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Final Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "imagenet100_expert.pth")))
model.eval()

# Final evaluation
val_loss, val_acc, val_top5 = evaluate(model, val_loader, criterion, device)

print("="*50)
print("Final Evaluation (Best Model)")
print("="*50)
print(f"Top-1 Accuracy: {val_acc:.2f}%")
print(f"Top-5 Accuracy: {val_top5:.2f}%")
print("="*50)

In [ ]:
# Visualize some predictions
images, labels = next(iter(val_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    _, predicted = outputs.max(1)

# Denormalize for visualization
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return tensor.cpu() * std + mean

class_names = train_dataset.classes

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).numpy().transpose(1, 2, 0)
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    
    true_label = class_names[labels[i]]
    pred_label = class_names[predicted[i]]
    color = 'green' if labels[i] == predicted[i] else 'red'
    
    ax.set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=9)
    ax.axis('off')

plt.suptitle("Sample Predictions (Green=Correct, Red=Wrong)", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("="*50)
print("ImageNet-100 Expert Training Complete!")
print("="*50)
print(f"Model: ResNet-18 (pretrained)")
print(f"Parameters: {total_params:,}")
print(f"")
print(f"Phase 1 (frozen): {PHASE1_EPOCHS} epochs, LR={PHASE1_LR}")
print(f"Phase 2 (fine-tune): {PHASE2_EPOCHS} epochs, LR={PHASE2_LR}")
print(f"")
print(f"Best Top-1 Accuracy: {best_acc:.2f}%")
print(f"Best Top-5 Accuracy: {val_top5:.2f}%")
print(f"")
print(f"Saved to: models/imagenet100_expert.pth")
print("="*50)
print("\nNext: Run 04_ImageNet100_Pipeline_Test.ipynb to test self-healing!")